# 05 — State & Memory: Knowledge Base Agent

**Module 4 of the workshop.** Code-defined store/retrieve routing — no framework magic, just an if/else and a local mem0/FAISS store. The simplest possible memory design, and the mental-model bridge into the fuller mem0 example.


## Problem

An LLM call is stateless by default — ask it something, get an answer, ask again with no memory of the first. Real assistants need to remember facts across separate invocations, not just within one conversation.


## Concept

```
Agent
 ├── Conversation state   (within one session)
 ├── Session               (one user's ongoing interaction)
 └── Long-term memory       (persists across sessions entirely)
```

Ask, for every memory design: *what does the agent remember, where is it stored, when does it disappear, how does a new call access it?* Here: facts persist in a local mem0/FAISS store, classified as store-vs-retrieve by a small classifier agent, and answered using only the retrieved context.

**What happens internally:** Store: content gets embedded (turned into a vector) and saved. Retrieve: the query gets embedded too, and the store returns whatever's closest by similarity — this is why the *embedding model* matters as much as the chat model.

**When would you NOT use this?** If you just need the last N messages of *this* conversation, that's session state, not memory — far simpler, no embeddings, no vector store. Reach for a memory store only when facts need to survive across sessions or be searched by meaning rather than recency.


## Architecture

```
"Remember that our office is closed on Fridays."
              │
              ▼
     ┌──────────────────┐
     │ determine_action  │──▶ classifier Agent (callback_handler=None)
     └────────┬──────────┘    "store" or "retrieve"?
              │
        action == "store"
              │
              ▼
     store.add(query, user_id="demo")   (mem0 → FAISS, mem0_data/faiss)
              │
              ▼
        "Stored." (returned directly, no LLM answer needed)


"When is the office closed?"
              │
              ▼
     ┌──────────────────┐
     │ determine_action  │──▶ classifier Agent
     └────────┬──────────┘    action == "retrieve"
              │
              ▼
     store.search(query, user_id="demo", limit=5)
              │
              ▼
        retrieved context
              │
              ▼
     answerer Agent(system_prompt="use ONLY the context")
              │
              ▼
        "The office is closed on Fridays." ──▶ user
```

Two agents besides the store itself: a **classifier** (decides store vs retrieve) and an **answerer** (constrained to only use retrieved context, not its own knowledge) — both silenced with `callback_handler=None` so only the final answer prints.


## Why mem0, not `strands_tools.memory`

`strands_tools.memory` is deprecated upstream, and its retrieve path returns nothing usable — a silent failure, not an error, so the agent just answers "I don't know" even for facts it actually stored. This script uses the same `mem0` + local FAISS store as `2-memory_agent.py` instead — real, working retrieval, still fully local and offline.

mem0 needs its own model config — it defaults to OpenAI internally unless told otherwise. `MEM0_CONFIG` below pins all three of mem0's moving parts to Ollama: `vector_store` (FAISS, on-disk), `embedder` (`nomic-embed-text`, text→vector), `llm` (`qwen3.5:4b`, mem0's own internal reasoning). Miss the `llm` block and it crashes looking for `OPENAI_API_KEY`.


## Step 1 — Resolve the model and set up the mem0 config

Note the dimension consistency: `embedding_model_dims=768` in the vector store config matches `embedding_dims=768` in the embedder config — both tied to `nomic-embed-text`'s real output size. A mismatch here is a classic silent-failure bug (wrong retrieval, not a crash).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from mem0 import Memory
from model_provider import OLLAMA_HOST, get_model
from strands import Agent

model = get_model()
print(f"Using: {type(model).__name__}")

MEM0_CONFIG = {
    "vector_store": {
        "provider": "faiss",
        "config": {"embedding_model_dims": 768, "path": "mem0_data/faiss"},
    },
    "embedder": {
        "provider": "ollama",
        "config": {"model": "nomic-embed-text", "ollama_base_url": OLLAMA_HOST, "embedding_dims": 768},
    },
    "llm": {
        "provider": "ollama",
        "config": {"model": "qwen3.5:4b", "ollama_base_url": OLLAMA_HOST},
    },
}

store = Memory.from_config(MEM0_CONFIG)


## Step 2 — Define the two system prompts

One classifies (`store` vs `retrieve`), one answers using only retrieved context — deliberately narrow prompts, not one do-everything prompt.


In [ ]:
ACTION_SYSTEM_PROMPT = """Classify the user's query as exactly one word: 'store' or
'retrieve'. 'store' = user wants to save/remember information.
'retrieve' = user is asking a question. Respond with only that one word."""

ANSWER_SYSTEM_PROMPT = """Answer the user's question using ONLY the provided context.
If the context doesn't contain the answer, say you don't know."""


## Step 3 — The routing function

`determine_action` is a tiny classifier agent — its only job is returning the word "store" or "retrieve", nothing else.


In [ ]:
def determine_action(query: str) -> str:
    classifier = Agent(model=model, system_prompt=ACTION_SYSTEM_PROMPT, callback_handler=None)
    return str(classifier(f"Query: {query}")).strip().lower()


## Step 4 — Store or retrieve, based on the classification

If storing: call `store.add()` directly and return a fixed confirmation, no LLM answer needed. If retrieving: fetch the closest-matching stored facts via `store.search()`, then let a constrained answerer agent phrase the response using *only* that context.


In [ ]:
def handle_query(query: str) -> str:
    action = determine_action(query)
    if "store" in action:
        store.add(query, user_id="demo")
        return "Stored."

    results = store.search(query, user_id="demo", limit=5)
    retrieved = "\n".join(r["memory"] for r in results.get("results", []))
    answerer = Agent(model=model, system_prompt=ANSWER_SYSTEM_PROMPT, callback_handler=None)
    return str(answerer(f"Context:\n{retrieved}\n\nQuestion: {query}"))


## Step 5 — Run it: store a fact, then retrieve it

The second call has no idea what happened in the first call except through the memory store — this is genuine persistence, not conversation history.


In [ ]:
print(handle_query("Remember that our office is closed on Fridays."))
print(handle_query("When is the office closed?"))


## Failure mode to know about

Retrieval returning "nothing relevant" isn't always a bug — if the stored fact and the query don't share enough semantic similarity, the agent correctly says "I don't know" about something that *is* in memory. Also watch for the OpenAI-credentials crash if the `llm` block ever gets dropped from `MEM0_CONFIG` — mem0 silently assumes OpenAI as its default internal model provider.
